# 配置输出的训练期OOF信息诊断
固定4项有向检验，原日线/实际执行H1收益，SOXL权重和现金权重。主统计控制年度与既定5/20趋势状态，63日均长平稳区块重抽样2000次，四检验双侧Bonferroni尾概率0.00625。不优化预测或交易。

In [1]:
from pathlib import Path
import os,json,hashlib
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from scipy.stats import spearmanr
from src.workflow.minute_action_value import _source
from src.objective.relative_wealth import stationary_block_bootstrap_indices
from src.data.nas_market_data import archive_nas_recorder
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/tangent_oof_information_20261015';plan=json.loads((card/'plan.json').read_text());provider=root/'data/qlib/full_path_tiered_allocation_20261013'
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(provider),region='us')
oof,oa=_source(root,plan['oof_source'],['analysis/oof_predictions']);training,ta=_source(root,plan['training_source'],['inputs/daily_frame','inputs/timing']);prices,pa=_source(root,plan['price_source'],['inputs/frame'])
pred=oof['analysis/oof_predictions'];frame=training['inputs/daily_frame'].reindex(pred.index);marks=prices['inputs/frame'].set_index('signal_date').reindex(pred.index);assert np.array_equal(pd.to_datetime(pred.trade_date),pd.to_datetime(marks.trade_date));assert pred.trade_date.max()<pd.Timestamp('2023-01-01')
d=pred[['soxl_weight','cash_weight','trade_date']].copy();d['year']=d.trade_date.dt.year;assert frame[['soxx_ret5','soxx_ret20']].notna().all().all();d['regime']=frame.soxx_ret5.gt(0).astype(int)+2*frame.soxx_ret20.gt(0).astype(int)
d['daily_relative']=frame.soxl_forward_open_return-frame.soxx_forward_open_return;d['daily_cash']=-frame.soxx_forward_open_return
d['execution_relative']=marks.soxl_exit_open/marks.soxl_open-marks.soxx_exit_open/marks.soxx_open;d['execution_cash']=1-marks.soxx_exit_open/marks.soxx_open
old_available=pd.to_datetime(training['inputs/timing'].reindex(pred.index).return_available_at);new_available=pd.to_datetime(marks.exit_date).dt.tz_localize('America/New_York')+pd.Timedelta(hours=9,minutes=49)
known=old_available.lt(pd.Timestamp(plan['cutoff']))&new_available.lt(pd.Timestamp(plan['cutoff']));targets=[pair[1] for pair in plan['tests']];d.loc[~known,targets]=np.nan;d['qualified_label']=known
# All calendar rows stay in the resampling axis; only missing labels are excluded inside statistics.
def conditional_stat(section,score,target):
 z=section[[score,target,'year','regime']].dropna();r=z.groupby(['year','regime'])[[score,target]].rank(pct=True,method='average');r=r-r.groupby([z.year,z.regime]).transform('mean')
 if len(r)<3 or r[score].std()==0 or r[target].std()==0:return np.nan
 return float(r[score].corr(r[target]))
indices=stationary_block_bootstrap_indices(observation_count=len(d),path_length=len(d),path_count=plan['bootstrap_resamples'],expected_block_length=plan['bootstrap_mean_block'],seed=plan['seed'])
rows=[];intervals=[]
for score,target in plan['tests']:
 for year,g in [('all',d),*list(d.groupby('year'))]:
  z=g[[score,target]].dropna();rows.append({'score':score,'target':target,'year':str(year),'n':len(z),'spearman':float(spearmanr(z[score],z[target]).statistic),'conditional_rank_correlation':conditional_stat(g,score,target)})
 values=np.array([conditional_stat(d.iloc[idx].reset_index(drop=True),score,target) for idx in indices]);valid=values[np.isfinite(values)];assert len(valid)==plan['bootstrap_resamples'];a=plan['two_sided_tail_probability'];lower,upper=np.quantile(valid,[a,1-a]);intervals.append({'score':score,'target':target,'n_calendar':len(d),'n_known':int(d[target].notna().sum()),'conditional_rank_correlation':conditional_stat(d,score,target),'lower':lower,'upper':upper,'lower_positive':bool(lower>0),'resamples':len(valid),'tail_probability':a})
table=pd.DataFrame(rows);ci=pd.DataFrame(intervals);policy={'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/daily','analysis/bootstrap_indices','analysis/tables']}
with R.start(experiment_name=card.name,uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan,source_audits={'oof':oa,'training':ta,'prices':pa});rec.save_objects(artifact_path='analysis',daily=d,bootstrap_indices=indices,tables={'statistics':table,'intervals':ci});rec.log_metrics(calendar_rows=len(d),known_rows=int(known.sum()),registered_artifact_count=5)
summary={'experiment':card.name,'decision':'diagnostic_only_no_promotion','new_fits':0,'new_backtests':0,'known_rows':int(known.sum()),'positive_lower_bounds':int(ci.lower_positive.sum()),'provider_uri':str(provider),'region':'us','as_of':'2026-10-09','recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(card/'qlib_recorder'),'recorder_local_dir':str(card/'qlib_recorder'/str(rec.experiment_id)/rec.id),'artifact_policy':policy,'objective_achieved':False,'scope':plan['scope']}
table.to_csv(card/'score_statistics.csv',index=False);ci.to_csv(card/'confidence_intervals.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'artifact_policy':policy}))
binding=json.loads((root/'research/opening_state_training_20261015/nas_binding.json').read_text());binding.pop('evidence',None);(card/'nas_binding.json').write_text(json.dumps(binding,indent=2))
a=Path(summary['recorder_local_dir'])/'artifacts';(card/'verification_manifest.json').write_text(json.dumps({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(a.rglob('*')) if p.is_file()]},indent=2))
summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));(card/'summary.json').write_text(json.dumps(summary,indent=2));print(ci.to_string(index=False))


# 配置层OOF信息诊断：四项条件信息区间均跨零

本轮0拟合、0新回测，复用此前9个年度模型的756日OOF配置预测。仅分析2020–2022交易年；两收益口径均成熟早于2023-01-01的755日进入统计，最后未知标签保留在重抽样日历中，不填为0。方向在运行前固定：SOXL权重应正向对应SOXL减SOXX次日收益，现金权重应正向对应负SOXX次日收益，现金利息按既定0处理。

主统计为年度×四种既定趋势状态内的百分位秩相关，趋势状态由信号日已知SOXX五日/二十日收益正负构成；各组秩中心化后合并计算相关。未来收益的分组排名只用于事后统计，绝不作为预测输入。固定均长63交易日的平稳区块重抽样2000次，每次重算组内秩；四项检验双侧Bonferroni尾概率0.00625（每项98.75%区间，名义家庭95%）。

| 输出与收益 | 条件秩相关 | 区块区间 |
|---|---:|---:|
| SOXL权重 / 原日线SOXL−SOXX | +0.0028 | [-0.0833, +0.0863] |
| 现金权重 / 原日线现金保护 | -0.0038 | [-0.0906, +0.0673] |
| SOXL权重 / 执行口径SOXL−SOXX | +0.0339 | [-0.0559, +0.1079] |
| 现金权重 / 执行口径现金保护 | +0.0051 | [-0.0846, +0.0875] |

四项下界均不为正。实际执行口径的年度SOXL条件相关2020/2021/2022为+0.0561/+0.0301/+0.0154，现金为+0.0352/+0.0126/-0.0328；年度表全部保留，没有挑表现好的年份。未控制年度/趋势时，执行口径全期Spearman分别0.0409、0.0468，也不应直接解释为交易超额。

结果只说明这些输出未通过预设的一日期限条件信息检验，不证明所有非线性关系、尾部保护或状态间风险控制均无效。条件控制会移除状态间信息；检验不含真实策略费用、换手和内生风险路径，因此不是完整收益检验。区块区间依赖弱平稳近似、有限样本和已选模型；四检验校正不覆盖此前全部研究搜索，OOF训练时序不等于研究选择独立。

没有改模型、翻转分数、切换预测期或寻找子集挽救。本轮解释了为何不能把击败固定配置对照当作已验证的预测alpha；最好的历史完整策略仍未获得稳定净SOXX超额。下一信息需求是能够在时间隔离检验中增加收益信息的新输入或有明确机制的标签设计，并先核查已关闭路线，不能继续只对现有弱信号做执行阈值优化或凭此诊断重新拟合。

决策diagnostic_only_no_promotion。模型和生产未变，目标未完成。Qlib来源身份、每日数据及重抽样索引存于Recorder，NAS逐对象冷恢复哈希通过，统计及区间表可复核。
